In [2]:
import os
import json
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer

W0922 15:59:05.836000 35356 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [9]:
# ==============================================================================
# ✨ 헬퍼 함수: Gemma 학습용 '대화 형식' 데이터셋 생성 ✨
# ==============================================================================
def create_gemma_dataset_chat_format(normal_path, phishing_path):
    """
    [함수 기능]
    Gemma를 대화 형식(Chat Format)으로 파인튜닝하기 위한 데이터셋을 생성합니다.
    이 함수 안에서 실제 파일 I/O(읽기)가 발생합니다.

    [파라미터 설명]
    - normal_path (str): 일반 통화 json 파일들이 들어있는 폴더 경로
    - phishing_path (str): 보이스피싱 json 파일들의 폴더 경로

    [사용 목적]
    모델이 'user'의 요청과 'assistant'의 답변 역할을 명확히 구분하여 학습하도록,
    구조화된 대화 형식의 데이터를 만드는 것이 핵심입니다.
    """
    data_list = []
    
    # Gemma에게 내릴 지시(Instruction) 템플릿
    instruction = "다음 대화 내용이 보이스피싱인지 아닌지 판단 후, '보이스피싱' 또는 '일반 대화'로만 답변해줘."

    # 1. 일반/피싱 데이터 로드 및 대화 형식 적용
    for label, path in [(0, normal_path), (1, phishing_path)]:
        answer = "일반 대화" if label == 0 else "보이스피싱"
        
        # os.listdir: 해당 폴더 내의 모든 파일 이름을 리스트로 가져옴
        for filename in os.listdir(path):
            if filename.endswith(".json"):
                # os.path.join: 폴더 경로와 파일 이름을 합쳐 완전한 파일 경로를 만듦
                file_path = os.path.join(path, filename)
                
                # with open(...): 파일을 열고, 다 읽으면 자동으로 닫아줌
                with open(file_path, 'r', encoding='utf-8') as f:
                    # json.load: 열린 json 파일의 내용을 파이썬 딕셔너리로 변환
                    json_data = json.load(f)
                
                # 각 json 파일 형식에 맞게 대화 내용(conversation) 추출
                if label == 0: # 일반 통화
                    dialogs = json_data['dataSet']['dialogs']
                    conversation = " ".join([d['text'] for d in dialogs])
                else: # 보이스피싱
                    conversation = json_data['text']
                
                # 'messages' 리스트 생성
                messages = [
                    {
                        "role": "user",
                        "content": f"{instruction}\n\n### 대화 내용:\n{conversation}"
                    },
                    {
                        "role": "assistant",
                        "content": answer
                    }
                ]
                # 최종 가공된 데이터를 data_list에 추가
                data_list.append({'messages': messages})
            
    # 모든 파일 처리가 끝나면, 리스트를 Pandas DataFrame으로 변환하여 반환
    return pd.DataFrame(data_list)

In [10]:
# ==============================================================================
# ✨ 메인 실행 부분 ✨
# ==============================================================================


# --- 1. 데이터 준비 ---
print("1. Gemma 파인튜닝용 '대화 형식' 데이터를 준비합니다.")

# ✨✨ 여기가 실제 데이터 로딩 및 가공이 시작되는 부분입니다. ✨✨
# 1-1. 사용자님의 데이터가 저장된 실제 폴더 경로를 변수에 할당합니다.
train_normal_dir = "./data/normal"
train_phishing_dir = "./data/phishing"

# 1-2. 위에서 정의한 헬퍼 함수에 폴더 경로를 전달하여 호출합니다.
#      이 함수 내부에서 모든 파일 읽기와 데이터 가공이 수행됩니다.
train_df = create_gemma_dataset_chat_format(train_normal_dir, train_phishing_dir)

# 1-3. 가공이 완료된 Pandas DataFrame을 Hugging Face의 Dataset 객체로 변환합니다.
#      이것이 모델 학습에 사용될 최종 데이터 형태입니다.
train_dataset = Dataset.from_pandas(train_df)

print(f"총 {len(train_dataset)}개의 학습 데이터 준비 완료.")
print("생성된 학습 데이터 예시 (첫 번째 샘플):")
# 가공된 데이터가 'messages' 형식으로 잘 만들어졌는지 직접 눈으로 확인
# print(train_dataset[0]['messages'])

# --- 2. 모델 및 토크나이저 로드 (4-bit QLoRA 적용) ---
print("2. Gemma 모델과 토크나이저를 로드합니다.")
model_name = "google/gemma-2b-it"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
# --- 3. LoRA 설정 ---
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# --- 4. 학습 설정 및 트레이너 실행 ---
training_args = TrainingArguments(
    output_dir="./gemma-results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    logging_steps=10,
    save_strategy="epoch"
)


trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    packing=True, 
)
print("파인튜닝을 시작합니다.")
trainer.train()

# --- 6. 학습된 어댑터 저장 ---
save_path = "./gemma-phishing-adapter"
print(f"파인튜닝된 LoRA 어댑터를 '{save_path}'에 저장합니다.")
trainer.save_model(save_path)
print("학습 및 저장이 완료")

1. Gemma 파인튜닝용 '대화 형식' 데이터를 준비합니다.
총 810개의 학습 데이터 준비 완료.
생성된 학습 데이터 예시 (첫 번째 샘플):
2. Gemma 모델과 토크나이저를 로드합니다.


c:\phishing_project\.venv\Lib\site-packages\accelerate\utils\modeling.py:1365: UserWarning: Current model requires 1152.0 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `load_in_8bit_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [11]:
print(train_dataset)

Dataset({
    features: ['messages'],
    num_rows: 810
})
